In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Okhla_Phase-2_Delhi_DPCC_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,364.0,168.0,206.0,120.0,179.0,252.0,96.0,51.0,92.0,133.0,329.0,295.0
1,2,373.0,263.0,115.0,127.0,160.0,164.0,114.0,62.0,78.0,165.0,327.0,283.0
2,3,365.0,206.0,114.0,178.0,258.0,141.0,101.0,62.0,81.0,154.0,400.0,264.0
3,4,397.0,279.0,118.0,182.0,269.0,218.0,54.0,55.0,72.0,212.0,385.0,170.0
4,5,360.0,196.0,118.0,160.0,269.0,269.0,69.0,49.0,71.0,144.0,383.0,145.0
5,6,346.0,135.0,110.0,153.0,227.0,191.0,56.0,64.0,101.0,130.0,349.0,169.0
6,7,363.0,153.0,154.0,151.0,314.0,245.0,48.0,67.0,55.0,118.0,391.0,232.0
7,8,384.0,147.0,124.0,164.0,236.0,249.0,44.0,58.0,84.0,144.0,396.0,319.0
8,9,394.0,116.0,121.0,234.0,173.0,152.0,85.0,54.0,100.0,139.0,355.0,151.0
9,10,306.0,297.0,153.0,251.0,163.0,143.0,125.0,50.0,118.0,132.0,331.0,258.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,364.000000,168.00000,206.000000,120.000000,179.0,252.000000,96.000000,51.000000,92.000000,133.0,329.000000,295.000000
1,2,373.000000,263.00000,115.000000,127.000000,160.0,164.000000,114.000000,62.000000,78.000000,165.0,327.000000,283.000000
2,3,365.000000,206.00000,114.000000,178.000000,258.0,141.000000,101.000000,62.000000,81.000000,154.0,400.000000,264.000000
3,4,397.000000,279.00000,118.000000,182.000000,269.0,218.000000,54.000000,55.000000,72.000000,212.0,385.000000,170.000000
4,5,360.000000,196.00000,118.000000,160.000000,269.0,269.000000,69.000000,49.000000,71.000000,144.0,383.000000,145.000000
5,6,346.000000,135.00000,110.000000,153.000000,227.0,191.000000,56.000000,64.000000,101.000000,130.0,349.000000,169.000000
6,7,363.000000,153.00000,154.000000,151.000000,314.0,245.000000,48.000000,67.000000,55.000000,118.0,391.000000,232.000000
7,8,384.000000,147.00000,124.000000,164.000000,236.0,249.000000,44.000000,58.000000,84.000000,144.0,396.000000,319.000000
8,9,394.000000,116.00000,121.000000,234.000000,173.0,152.000000,85.000000,54.000000,100.000000,139.0,355.000000,151.000000
9,10,306.000000,297.00000,153.000000,170.666667,163.0,143.000000,125.000000,50.000000,118.000000,132.0,331.000000,258.000000
